# FD001 Boosting Models

Objetivo: comparar la baseline `temporal_w30 + RandomForestRegressor` contra modelos tabulares fuertes usando solo validación artificial por motores y cutoffs. No se usa el test oficial FD001.


In [ ]:
from collections import OrderedDict
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
PREDICTIONS_DIR = PROJECT_ROOT / "predictions"
for path in [RESULTS_DIR, FIGURES_DIR, PREDICTIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SEED = 42
DATA_DIR = PROJECT_ROOT / "CMAPSSData"
EVAL_SIZE = 0.2
CUT_RULS = (20, 50, 80, 110, 140)
DEFAULT_WINDOW_SIZE = 30
DEFAULT_RUL_CAP = 125

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)


In [ ]:
from src.preprocessed_FD001 import (
    prepare_fd001_temporal_validation_only,
    preprocessing_summary,
)
from src.fd001_modeling import (
    metrics_by_model,
    metrics_by_rul_bin,
    plot_validation_diagnostics,
    prediction_frame,
    regression_metrics,
    set_global_seed,
)


def add_bin_metric_columns(metrics, bin_metrics):
    result = metrics.copy()
    for label in ["0-30", "30-60", "60-90", "90+"]:
        safe_label = label.replace("-", "_").replace("+", "plus")
        subset = bin_metrics.loc[bin_metrics["rul_bin"].astype(str) == label]
        for _, row in subset.iterrows():
            mask = (
                (result["representation"] == row["representation"])
                & (result["model_name"] == row["model_name"])
            )
            result.loc[mask, f"mae_rul_{safe_label}"] = row["mae"]
            result.loc[mask, f"dangerous_error_pct_rul_{safe_label}"] = row["dangerous_error_pct"]
    return result


def available_tabular_factories(random_state=42):
    from sklearn.ensemble import (
        ExtraTreesRegressor,
        GradientBoostingRegressor,
        HistGradientBoostingRegressor,
        RandomForestRegressor,
    )

    factories = OrderedDict()
    availability = []
    has_external_boosting = False

    factories["RandomForestRegressor"] = lambda: RandomForestRegressor(
        n_estimators=250,
        max_depth=14,
        min_samples_leaf=3,
        random_state=random_state,
        n_jobs=-1,
    )

    try:
        from xgboost import XGBRegressor
        factories["XGBRegressor"] = lambda: XGBRegressor(
            n_estimators=160,
            max_depth=3,
            learning_rate=0.04,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="reg:squarederror",
            random_state=random_state,
            n_jobs=-1,
            tree_method="hist",
            verbosity=0,
        )
        has_external_boosting = True
        availability.append("XGBoost disponible: se incluye XGBRegressor.")
    except Exception as exc:
        availability.append(f"XGBoost no disponible: {type(exc).__name__}.")

    try:
        from lightgbm import LGBMRegressor
        factories["LGBMRegressor"] = lambda: LGBMRegressor(
            n_estimators=220,
            max_depth=-1,
            num_leaves=31,
            learning_rate=0.035,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=0.1,
            random_state=random_state,
            n_jobs=-1,
            verbose=-1,
        )
        has_external_boosting = True
        availability.append("LightGBM disponible: se incluye LGBMRegressor.")
    except Exception as exc:
        availability.append(f"LightGBM no disponible: {type(exc).__name__}.")

    if not has_external_boosting:
        availability.append("Sin XGBoost/LightGBM: se usan fallbacks sklearn.")
        factories["HistGradientBoostingRegressor"] = lambda: HistGradientBoostingRegressor(
            max_iter=180,
            learning_rate=0.05,
            max_leaf_nodes=31,
            l2_regularization=0.01,
            random_state=random_state,
        )
        factories["GradientBoostingRegressor"] = lambda: GradientBoostingRegressor(
            n_estimators=160,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.85,
            random_state=random_state,
        )
        factories["ExtraTreesRegressor"] = lambda: ExtraTreesRegressor(
            n_estimators=220,
            max_depth=16,
            min_samples_leaf=2,
            random_state=random_state,
            n_jobs=-1,
        )
    return factories, availability

def fit_predict_model(prepared, model_name, model, representation, sample_weight=None):
    if sample_weight is None:
        model.fit(prepared["X_train"], prepared["y_train"])
    else:
        model.fit(prepared["X_train"], prepared["y_train"], sample_weight=sample_weight)
    preds = model.predict(prepared["X_eval"])
    return prediction_frame(
        prepared["eval_df"],
        preds,
        model_name=model_name,
        representation=representation,
    ), model


## Preparación

Se usan features temporales tabulares resumidas, no ventanas crudas como secuencia. El scaler se ajusta solo con los motores de train.


In [ ]:
set_global_seed(SEED)
prepared = prepare_fd001_temporal_validation_only(
    data_dir=DATA_DIR,
    eval_size=EVAL_SIZE,
    random_state=SEED,
    max_rul=DEFAULT_RUL_CAP,
    cut_ruls=CUT_RULS,
    window_size=DEFAULT_WINDOW_SIZE,
)
preprocessing_summary(prepared)


## Modelos

Si XGBoost o LightGBM no están instalados, el notebook sigue con fallbacks de sklearn.


In [ ]:
model_factories, availability_notes = available_tabular_factories(random_state=SEED)
print("\n".join(availability_notes))
list(model_factories.keys())


## Entrenamiento y validación artificial


In [ ]:
representation = f"temporal_w{DEFAULT_WINDOW_SIZE}"
prediction_tables = []
fitted_models = {}

for model_name, factory in model_factories.items():
    print(f"Entrenando {model_name}...")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pred_df, model = fit_predict_model(prepared, model_name, factory(), representation)
    prediction_tables.append(pred_df)
    fitted_models[model_name] = model

predictions = pd.concat(prediction_tables, ignore_index=True)
metrics = metrics_by_model(predictions)
metrics.insert(2, "window_size", DEFAULT_WINDOW_SIZE)
metrics.insert(3, "rul_cap", DEFAULT_RUL_CAP)
metrics.insert(4, "n_features", len(prepared["feature_columns"]))
metrics.insert(5, "target_used_for_training", "RUL_capped")

bin_metrics = metrics_by_rul_bin(predictions)
metrics = add_bin_metric_columns(metrics, bin_metrics)

metrics


## Guardado


In [ ]:
figure_dir = FIGURES_DIR / "fd001_boosting"
figure_dir.mkdir(parents=True, exist_ok=True)

metrics.to_csv(RESULTS_DIR / "fd001_boosting_metrics.csv", index=False)
predictions.to_csv(RESULTS_DIR / "fd001_boosting_predictions.csv", index=False)
bin_metrics.to_csv(RESULTS_DIR / "fd001_boosting_metrics_by_rul_bin.csv", index=False)
plot_validation_diagnostics(predictions, figure_dir, "FD001 boosting validation")

print("Guardado:")
print(RESULTS_DIR / "fd001_boosting_metrics.csv")
print(RESULTS_DIR / "fd001_boosting_predictions.csv")
print(figure_dir)


## Lectura rápida


In [ ]:
display(metrics.sort_values(["rmse", "mae"]))
display(bin_metrics.sort_values(["rul_bin", "rmse"]))
